In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Setup Tools


In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [4]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [5]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [6]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str = ""
    destination: str = ""
    guest_count: str = ""
    genre: str = ""
    wedding_date: str = "" # New field

## Create Subagents


In [7]:
from langchain.agents import create_agent

# Travel agent

from datetime import datetime

current_date = datetime.now().strftime("%Y-%m-%d")

# Travel agent
travel_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=f"""
    You are a travel agent. Today's date is {current_date}.
    
    Your task is to search for flights for a wedding on the SPECIFIC DATE provided by the coordinator.
    
    CRITICAL RULES:
    1. The 'departureDate' MUST be exactly the date provided in the request.
    2. If no date is provided, choose a reasonable date at least 2 months in the future.
    3. The date MUST be in YYYY-MM-DD format.
    4. NEVER search for a date before {current_date}.
    """
)

In [8]:
# Venue agent
venue_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    """
)

In [9]:
# Playlist agent
playlist_agent = create_agent(
    model=model,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.
    You may need to make multiple queries to iteratively find the best options.
    """
)

## Main Coordinator


In [10]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

# Use .get() in the tools to avoid KeyError

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights. Uses the wedding_date from the state."""
    origin = runtime.state.get("origin")
    destination = runtime.state.get("destination")
    wedding_date = runtime.state.get("wedding_date")
    
    if not origin or not destination or not wedding_date:
        return "Error: Missing origin, destination, or wedding_date. Use update_state first."
    
    query = f"Find flights from {origin} to {destination} departing on {wedding_date}"
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content


@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    # Using .get() with a fallback or check
    destination = runtime.state.get("destination")
    capacity = runtime.state.get("guest_count")
    
    if not destination or not capacity:
        return "Error: Destination or guest_count is missing from state. Use update_state first."
        
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content


@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state.get("genre")
    
    if not genre:
        return "Error: Genre is missing from state. Use update_state first."
        
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content


@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, wedding_date: str, runtime: ToolRuntime) -> str:
    """Update the state with all wedding details, including the specific wedding_date (YYYY-MM-DD)."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "wedding_date": wedding_date,
        "messages": [ToolMessage("State updated with date.", tool_call_id=runtime.tool_call_id)]
    })


In [11]:
from langchain.agents import create_agent
from datetime import datetime
current_date = datetime.now().strftime("%A, %B %d, %Y")

coordinator = create_agent(
    model=model,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt=f"""
    You are a wedding coordinator. Today is {current_date}.
    
    TASK:
    1. Extract the origin, destination, guest count, and genre.
    2. Determine the wedding date:
       - If the user provides a specific date, use it.
       - If the user says "in two months," calculate the date relative to {current_date}.
       - If no date is mentioned, default to exactly 6 months from today.
    3. Format the date strictly as YYYY-MM-DD.
    4. Call 'update_state' with ALL these values immediately.
    5. After the state is updated, call your specialists to get flight, venue, and music options.
    6. Summarize the full wedding plan for the user.
    """
)

## Test


In [ ]:
from langchain_core.messages import HumanMessage

# Define the initial state with empty values to avoid potential KeyErrors
initial_input = {
    "messages": [
        HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre, in about three months")
    ],
    "origin": "",
    "destination": "",
    "guest_count": "",
    "genre": "",
    "wedding_date": ""
}

# Run the coordinator
response = await coordinator.ainvoke(initial_input)

# Cleanly print the final decision
print("--- WEDDING PLAN ---")
print(response["messages"][-1].content)

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r